[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/39_ml_systems/02_reproducibility_and_experiment_tracking.ipynb)

# Bài 02 — Tái lập và theo dõi thí nghiệm

**🎯 Mục tiêu**
- Biết các nguồn khiến kết quả ML **không tái lập được**: ngẫu nhiên, thứ tự dữ liệu, số thực dấu phẩy động, môi trường, dữ liệu thay đổi.
- Kiểm soát seed đúng cách; đo **độ biến thiên theo seed** thay vì tin một con số; so sánh mô hình **theo cặp**.
- Ghi lại môi trường, **dấu vân tay dữ liệu** (hash) và cấu hình — đủ để chạy lại đúng một run.
- Tự viết một **experiment tracker** tối giản, rồi dùng MLflow cho cùng việc đó; nhận ra **lời nguyền người thắng** khi chọn cấu hình trên tập validation.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/39_ml_systems

In [ ]:
import hashlib
import json
import math
import os
import platform
import random
import subprocess
import sys
import tempfile
import time
import uuid
import warnings
from contextlib import contextmanager
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from ml_course.datasets import make_customer_churn

## 1. "Tái lập" nghĩa là gì?

| Mức | Câu hỏi | Ví dụ |
|---|---|---|
| **Lặp lại** (repeatable) | Chạy lại trên **máy mình** có ra đúng kết quả? | Notebook chạy lại sau 1 tháng cho cùng AUC |
| **Tái lập** (reproducible) | **Người khác**, máy khác, từ code + dữ liệu + cấu hình của mình có ra cùng kết quả? | Đồng nghiệp chạy lại run "tốt nhất" trong báo cáo |
| **Tái hiện** (replicable) | Trên **dữ liệu mới**, kết luận còn đúng? | Mô hình B tốt hơn A trên dữ liệu tháng sau |

Mỗi kết quả là hàm của **code × dữ liệu × cấu hình × môi trường × ngẫu nhiên**. Thiếu bất kỳ thành phần nào, ta không chạy lại được. Bảng các nguồn biến thiên thường gặp:

| Nguồn | Ví dụ | Kiểm soát |
|---|---|---|
| Ngẫu nhiên | khởi tạo trọng số, bootstrap, chia train/test, dropout | seed tường minh cho **mọi** thành phần |
| Thứ tự | thứ tự dòng, thứ tự phần tử của `set`/`dict`, thứ tự file trong thư mục | sắp xếp tường minh |
| Số thực | thứ tự cộng khi chạy song song (đa luồng, GPU) | thuật toán tất định, cố định số luồng, so sánh có dung sai |
| Môi trường | phiên bản thư viện, phần cứng, hệ điều hành | lock file, Docker (chương 40), ghi lại phiên bản |
| Dữ liệu | bảng nguồn được cập nhật, sửa lỗi, thêm dòng | snapshot + hash dữ liệu (DVC, bảng có phiên bản) |
| Code | sửa code sau khi chạy | commit trước khi chạy, ghi commit hash vào run |

## 2. Ngẫu nhiên và seed

In [ ]:
raw = make_customer_churn(3000, seed=0)
X = pd.get_dummies(raw.drop(columns=["customer_id", "churn"]), dtype=float)
y = raw.churn.values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)


def fit_auc(model, X_tr=X_tr, y_tr=y_tr, X_te=X_te, y_te=y_te):
    return roc_auc_score(y_te, model.fit(X_tr, y_tr).predict_proba(X_te)[:, 1])


print("random_state=None:", [round(fit_auc(RandomForestClassifier(100, n_jobs=1)), 4) for _ in range(3)])
print("random_state=0   :", [round(fit_auc(RandomForestClassifier(100, random_state=0, n_jobs=1)), 4) for _ in range(3)])
seed_aucs = [fit_auc(RandomForestClassifier(100, random_state=s, n_jobs=1)) for s in range(10)]
print(f"10 seed mô hình khác nhau: AUC từ {min(seed_aucs):.4f} đến {max(seed_aucs):.4f}")

Cố định seed làm kết quả **lặp lại được**, nhưng không làm nó **đúng hơn**: seed chỉ chọn một điểm trong phân phối kết quả. Chênh lệch giữa các seed là **nhiễu** — mọi cải tiến nhỏ hơn mức nhiễu này đều đáng ngờ.

Quy tắc thực hành:
- Truyền seed **tường minh** vào từng thành phần (`random_state=`, `np.random.default_rng(seed)`, `torch.Generator`), thay vì dựa vào trạng thái toàn cục.
- Một hàm `set_seed` cho các thư viện vẫn dùng trạng thái toàn cục (`random`, `np.random`, PyTorch — chương 28 có thêm cờ `cudnn.deterministic`).

In [ ]:
def set_seed(seed: int) -> np.random.Generator:
    """Đặt các nguồn ngẫu nhiên toàn cục; trả về Generator riêng để dùng tường minh."""
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)                                         # seed cả CPU và mọi GPU
    except ImportError:
        pass
    return np.random.default_rng(seed)


rng = set_seed(42)
print(rng.integers(0, 100, 5), set_seed(42).integers(0, 100, 5))

Một nguồn ngẫu nhiên **ẩn** ít ai để ý: Python băm chuỗi với một seed ngẫu nhiên mỗi lần khởi động (`PYTHONHASHSEED`), nên **thứ tự duyệt `set` các chuỗi** thay đổi giữa các lần chạy. Nếu danh sách cột đặc trưng được tạo từ một `set`, thứ tự cột — và do đó mô hình — đổi theo mỗi lần chạy.

In [ ]:
code = "print(list({'tenure', 'charges', 'calls', 'contract', 'products'}))"
for hash_seed in ["1", "2", "3"]:
    out = subprocess.run([sys.executable, "-c", code], env={**os.environ, "PYTHONHASHSEED": hash_seed},
                         capture_output=True, text=True).stdout.strip()
    print(f"PYTHONHASHSEED={hash_seed}: {out}")
print("cách đúng — sắp xếp tường minh:", sorted({'tenure', 'charges', 'calls', 'contract', 'products'}))

## 3. Đừng tin một lần chia: so sánh mô hình theo cặp

Hai mô hình có hiệu năng gần nhau. Ta lặp lại 20 lần chia train/test khác nhau và xem "ai thắng".

In [ ]:
candidates = {
    "HGB": lambda: HistGradientBoostingClassifier(learning_rate=0.03, max_leaf_nodes=6, max_iter=200, random_state=0),
    "RF": lambda: RandomForestClassifier(200, min_samples_leaf=20, random_state=0, n_jobs=1),
}
rows = []
for split_seed in range(20):
    a_tr, a_te, b_tr, b_te = train_test_split(X, y, test_size=0.3, random_state=split_seed, stratify=y)
    rows.append({name: fit_auc(make(), a_tr, b_tr, a_te, b_te) for name, make in candidates.items()})
split_results = pd.DataFrame(rows)
diff = split_results.HGB - split_results.RF
print(f"HGB thắng {int((diff > 0).sum())}/20 lần chia, RF thắng {int((diff < 0).sum())}/20")
print(f"AUC HGB = {split_results.HGB.mean():.4f} ± {split_results.HGB.std():.4f} | RF = {split_results.RF.mean():.4f} ± {split_results.RF.std():.4f}")
lo, hi = stats.t.interval(0.95, len(diff) - 1, loc=diff.mean(), scale=diff.std(ddof=1) / np.sqrt(len(diff)))
print(f"chênh lệch theo cặp HGB − RF = {diff.mean():+.4f} (độ lệch chuẩn {diff.std():.4f}), KTC 95% [{lo:+.4f}, {hi:+.4f}]")

In [ ]:
import matplotlib.pyplot as plt

from ml_course.visualization import set_style

set_style()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(split_results.RF, split_results.HGB, color="C0")
lims = [split_results.min().min() - 0.003, split_results.max().max() + 0.003]
axes[0].plot(lims, lims, "k--", lw=1, label="hai mô hình bằng nhau")
axes[0].set_xlabel("ROC-AUC của RF")
axes[0].set_ylabel("ROC-AUC của HGB")
axes[0].set_title("Mỗi điểm là một lần chia train/test")
axes[0].legend()
axes[1].hist(diff, bins=12, color="C2")
axes[1].axvline(0, color="k", ls="--")
axes[1].set_xlabel("HGB − RF trên cùng lần chia")
axes[1].set_title("Chênh lệch theo cặp")
plt.tight_layout()
plt.show()

- Với **một** lần chia, kết luận "HGB tốt hơn" hay "RF tốt hơn" phụ thuộc may rủi.
- Độ lệch chuẩn AUC **giữa các lần chia** (~0.01) lớn hơn nhiều chênh lệch giữa hai mô hình; nhưng độ lệch chuẩn của **chênh lệch theo cặp** (cùng lần chia cho cả hai) nhỏ hơn hẳn — vì cái khó/dễ của từng tập test ảnh hưởng **cả hai** mô hình như nhau. Luôn so sánh trên **cùng** các lần chia/fold.
- Khoảng tin cậy chứa 0 → không có bằng chứng mô hình nào tốt hơn; chọn mô hình đơn giản/rẻ hơn. (Lưu ý: các lần chia lặp lại dùng chung dữ liệu nên không độc lập hoàn toàn — KTC này hơi hẹp hơn thực tế; xem corrected resampled t-test của Nadeau & Bengio.)

## 4. Số thực dấu phẩy động: thứ tự phép cộng

In [ ]:
print("(0.1 + 0.2) + 0.3 == 0.1 + (0.2 + 0.3)?", (0.1 + 0.2) + 0.3 == 0.1 + (0.2 + 0.3))
v = np.random.default_rng(0).normal(size=1_000_000).astype(np.float32)
print(f"tổng float32 — xuôi: {np.sum(v):.4f} | ngược: {np.sum(v[::-1]):.4f} | đã sắp xếp: {np.sum(np.sort(v)):.4f} | chính xác (fsum, float64): {math.fsum(v.astype(np.float64)):.4f}")

Phép cộng số thực **không kết hợp**. Khi tính song song (nhiều luồng CPU, GPU dùng atomic add), thứ tự cộng thay đổi giữa các lần chạy → sai khác nhỏ → bị khuếch đại qua hàng nghìn bước huấn luyện (một ngưỡng chia cây đổi, một argmax đổi). Hệ quả thực hành:
- Muốn **lặp lại bit-by-bit**: cố định số luồng, bật thuật toán tất định (`torch.use_deterministic_algorithms(True)`), cùng phần cứng và phiên bản thư viện — thường chậm hơn.
- Trong kiểm thử: so sánh **có dung sai** (`np.allclose`), không dùng `==` cho số thực.

## 5. Ghi lại môi trường

In [ ]:
def capture_environment(packages=("numpy", "pandas", "scipy", "scikit-learn", "mlflow", "torch")) -> dict:
    """Ảnh chụp môi trường: Python, hệ điều hành, phiên bản thư viện, commit git (nếu có)."""
    versions = {}
    for name in packages:
        try:
            versions[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            versions[name] = None
    try:
        commit = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, timeout=5).stdout.strip() or None
    except (OSError, subprocess.SubprocessError):
        commit = None
    return {"python": platform.python_version(), "platform": platform.platform(), "cpu_count": os.cpu_count(),
            "packages": versions, "git_commit": commit, "timestamp_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}


env_snapshot = capture_environment()
print(json.dumps({k: v for k, v in env_snapshot.items() if k != "timestamp_utc"}, indent=2, ensure_ascii=False))

Ghi phiên bản giúp **chẩn đoán**; muốn **tái tạo** môi trường cần thêm:
- **Lock file** ghim chính xác mọi gói (kể cả phụ thuộc gián tiếp): `pip freeze > requirements.lock`, `uv lock`, `poetry.lock`, `conda env export`.
- **Docker image** đóng băng cả hệ điều hành và thư viện hệ thống (chương 40).
- `git_commit` = `None` nghĩa là thư mục không phải repo git — trong dự án thật, hãy **commit trước khi chạy** và từ chối chạy thí nghiệm "chính thức" khi còn thay đổi chưa commit.

## 6. Dấu vân tay dữ liệu

"Dữ liệu train" thường là một truy vấn tới kho dữ liệu đang được cập nhật. Ghi **hash nội dung** dữ liệu vào mỗi run để biết chắc hai run có dùng cùng dữ liệu hay không.

In [ ]:
def fingerprint(df: pd.DataFrame, row_order_matters: bool = True) -> str:
    """SHA-256 (rút gọn) của nội dung DataFrame: tên cột + giá trị từng dòng."""
    df = df.sort_index(axis=1)                                          # thứ tự cột không quan trọng
    row_hashes = pd.util.hash_pandas_object(df, index=False).values    # một hash 64-bit cho mỗi dòng
    if not row_order_matters:
        row_hashes = np.sort(row_hashes)
    h = hashlib.sha256()
    h.update("|".join(f"{c}:{t}" for c, t in df.dtypes.items()).encode())
    h.update(row_hashes.tobytes())
    return h.hexdigest()[:16]


base = raw.copy()
edited = raw.copy()
edited.loc[123, "monthly_charges"] += 0.01                             # sửa MỘT giá trị
shuffled = raw.sample(frac=1, random_state=0)
print("gốc                  :", fingerprint(base))
print("chạy lại             :", fingerprint(raw.copy()))
print("sửa 1 ô              :", fingerprint(edited))
print("đảo dòng             :", fingerprint(shuffled), "| bỏ qua thứ tự dòng:", fingerprint(shuffled, row_order_matters=False), "=", fingerprint(base, row_order_matters=False))
print("file CSV (sha256)    :", hashlib.sha256(Path("../../data/sample/customer_churn.csv").read_bytes()).hexdigest()[:16])

- Hash đổi khi **một ô** đổi → phát hiện dữ liệu "lặng lẽ" thay đổi giữa hai run.
- Thứ tự dòng có quan trọng không là **quyết định thiết kế**: với chia train/test theo `random_state`, đảo dòng làm đổi tập train → cần hash có thứ tự.
- Công cụ: **DVC** lưu dữ liệu lớn ở kho ngoài (S3, GCS…) và commit một file nhỏ chứa hash vào git; **Delta Lake/Iceberg** có "time travel" theo phiên bản bảng; feature store ghi phiên bản đặc trưng.

**DVC — quy trình tối thiểu** (cần `pip install dvc`; chạy trong một repo git, không chạy trong bài):

```bash
dvc init                                    # một lần cho mỗi repo
dvc add data/raw/transactions.csv           # tạo transactions.csv.dvc (chứa hash md5) và đưa file gốc vào .gitignore
git add data/raw/transactions.csv.dvc data/raw/.gitignore
git commit -m "dữ liệu v1"
dvc remote add -d storage s3://my-bucket/dvc    # kho dữ liệu: S3, GCS, Azure, SSH, ổ mạng…
dvc push                                    # đẩy dữ liệu lên kho
# … dữ liệu thay đổi → dvc add lại → commit file .dvc mới (dữ liệu v2)
git checkout <commit-cũ> && dvc checkout    # lấy lại ĐÚNG phiên bản dữ liệu của commit đó
dvc repro                                   # chạy lại pipeline (dvc.yaml) khi dữ liệu hoặc code đổi
```

File `.dvc` nhỏ chỉ chứa hash — giống hàm `fingerprint` ở trên — nên git quản lý được "phiên bản dữ liệu" song song với phiên bản code, còn dữ liệu thật nằm ở kho ngoài.

## 7. Tự viết experiment tracker

Mỗi run cần lưu: **cấu hình** (params), **kết quả** (metrics), **ngữ cảnh** (môi trường, hash dữ liệu, commit), **sản phẩm** (mô hình, biểu đồ), trạng thái. Tracker tối giản dưới đây lưu mỗi run thành một thư mục chứa `run.json`.

In [ ]:
class Run:
    def __init__(self, root: Path, name: str | None):
        self.dir = root / uuid.uuid4().hex[:8]
        self.dir.mkdir(parents=True)
        self.record = {"run_id": self.dir.name, "name": name, "status": "RUNNING",
                       "start": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                       "params": {}, "metrics": {}, "tags": {}, "artifacts": []}
        self._save()

    def _save(self):                                                    # ghi ngay mỗi lần log → run bị lỗi giữa chừng vẫn còn dấu vết
        (self.dir / "run.json").write_text(json.dumps(self.record, indent=2, ensure_ascii=False), encoding="utf-8")

    def log_params(self, params: dict):
        self.record["params"].update(params)
        self._save()

    def log_metrics(self, metrics: dict):
        self.record["metrics"].update({k: float(v) for k, v in metrics.items()})
        self._save()

    def set_tags(self, tags: dict):
        self.record["tags"].update(tags)
        self._save()

    def log_artifact(self, name: str, content: str):
        (self.dir / name).write_text(content, encoding="utf-8")
        self.record["artifacts"].append(name)
        self._save()


class LocalTracker:
    def __init__(self, root):
        self.root = Path(root)
        self.root.mkdir(parents=True, exist_ok=True)

    @contextmanager
    def start_run(self, name: str | None = None):
        run = Run(self.root, name)
        try:
            yield run
            run.record["status"] = "FINISHED"
        except Exception:
            run.record["status"] = "FAILED"
            raise
        finally:
            run.record["end"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
            run._save()

    def search(self) -> pd.DataFrame:
        rows = []
        for path in self.root.glob("*/run.json"):
            r = json.loads(path.read_text(encoding="utf-8"))
            rows.append({"run_id": r["run_id"], "name": r["name"], "status": r["status"], "start": r["start"],
                         **{f"params.{k}": v for k, v in r["params"].items()},
                         **{f"metrics.{k}": v for k, v in r["metrics"].items()},
                         **{f"tags.{k}": v for k, v in r["tags"].items()}})
        return pd.DataFrame(rows).sort_values("start", kind="stable").reset_index(drop=True)

Huấn luyện được viết thành **một hàm thuần** của cấu hình: mọi thứ ảnh hưởng tới kết quả đều nằm trong `TrainConfig` — kể cả seed dữ liệu và seed chia tập.

In [ ]:
@dataclass(frozen=True)
class TrainConfig:
    learning_rate: float = 0.1
    max_leaf_nodes: int = 31
    min_samples_leaf: int = 20
    max_iter: int = 100
    model_seed: int = 0
    data_seed: int = 0
    split_seed: int = 0
    n_rows: int = 3000


def train(config: TrainConfig):
    data = make_customer_churn(config.n_rows, seed=config.data_seed)
    features = pd.get_dummies(data.drop(columns=["customer_id", "churn"]), dtype=float)
    f_tr, f_va, t_tr, t_va = train_test_split(features, data.churn.values, test_size=0.3,
                                              random_state=config.split_seed, stratify=data.churn.values)
    model = HistGradientBoostingClassifier(learning_rate=config.learning_rate, max_leaf_nodes=config.max_leaf_nodes,
                                           min_samples_leaf=config.min_samples_leaf, max_iter=config.max_iter,
                                           random_state=config.model_seed).fit(f_tr, t_tr)
    metrics = {"val_auc": roc_auc_score(t_va, model.predict_proba(f_va)[:, 1]),
               "train_auc": roc_auc_score(t_tr, model.predict_proba(f_tr)[:, 1])}
    return model, metrics, fingerprint(data)


tracker = LocalTracker(Path(tempfile.mkdtemp()) / "runs")
for lr in [0.03, 0.1, 0.3]:
    for leaves in [8, 31]:
        config = TrainConfig(learning_rate=lr, max_leaf_nodes=leaves)
        with tracker.start_run(name=f"hgb_lr{lr}_leaf{leaves}") as run:
            t0 = time.perf_counter()
            run.log_params(asdict(config))
            model, metrics, data_hash = train(config)
            run.log_metrics({**metrics, "train_seconds": time.perf_counter() - t0})
            run.set_tags({"data_hash": data_hash, "git_commit": str(env_snapshot["git_commit"]),
                          "sklearn": env_snapshot["packages"]["scikit-learn"]})
            run.log_artifact("environment.json", json.dumps(env_snapshot, indent=2))

runs = tracker.search()
cols = ["name", "params.learning_rate", "params.max_leaf_nodes", "metrics.train_auc", "metrics.val_auc", "tags.data_hash"]
print(runs[cols].sort_values("metrics.val_auc", ascending=False).round(4).to_string(index=False))

Phép thử quan trọng nhất của tracker: **từ những gì đã log, chạy lại được đúng run đó**.

In [ ]:
best = runs.sort_values("metrics.val_auc", ascending=False).iloc[0]
logged = json.loads((tracker.root / best.run_id / "run.json").read_text(encoding="utf-8"))
replayed_config = TrainConfig(**logged["params"])                       # dựng lại cấu hình CHỈ từ file đã log
_, replayed_metrics, replayed_hash = train(replayed_config)
print(f"run {best.run_id}: val_auc đã log = {logged['metrics']['val_auc']:.6f} | chạy lại = {replayed_metrics['val_auc']:.6f}")
print("cùng dữ liệu:", replayed_hash == logged["tags"]["data_hash"], "| khớp tuyệt đối:", replayed_metrics["val_auc"] == logged["metrics"]["val_auc"])

## 8. Lời nguyền người thắng (winner's curse)

Khi thử **nhiều** cấu hình rồi chọn cái có điểm validation cao nhất, điểm validation của "người thắng" bị **lạc quan**: nó tốt một phần nhờ thật sự tốt, một phần nhờ **may** trên tập validation cụ thể đó. Mô phỏng: huấn luyện 30 cấu hình ngẫu nhiên một lần, rồi lặp 200 lần việc rút một tập validation mới, chọn cấu hình tốt nhất, và so điểm validation của nó với AUC "thật" (trên 17.000 khách hàng).

In [ ]:
big = make_customer_churn(20000, seed=0)
X_big = pd.get_dummies(big.drop(columns=["customer_id", "churn"]), dtype=float)
y_big = big.churn.values
order = np.random.default_rng(0).permutation(len(big))
train_idx, pool_idx = order[:3000], order[3000:]
search_rng = np.random.default_rng(0)
proba = []
for _ in range(30):
    params = dict(learning_rate=float(10 ** search_rng.uniform(-2, -0.5)), max_leaf_nodes=int(search_rng.integers(4, 64)),
                  min_samples_leaf=int(search_rng.integers(5, 100)), max_iter=int(search_rng.integers(50, 200)))
    model = HistGradientBoostingClassifier(random_state=0, **params).fit(X_big.iloc[train_idx], y_big[train_idx])
    proba.append(model.predict_proba(X_big)[:, 1])
proba = np.array(proba)
true_auc = np.array([roc_auc_score(y_big[pool_idx], p[pool_idx]) for p in proba])

curse_rows = []
for n_val in [200, 400, 2000]:
    best_val, chosen_true = [], []
    for _ in range(200):
        val = search_rng.choice(pool_idx, n_val, replace=False)
        val_auc = np.array([roc_auc_score(y_big[val], p[val]) for p in proba])
        best_val.append(val_auc.max())
        chosen_true.append(true_auc[val_auc.argmax()])
    curse_rows.append({"cỡ validation": n_val, "AUC val của cấu hình thắng": np.mean(best_val),
                       "AUC thật của nó": np.mean(chosen_true), "độ lạc quan": np.mean(best_val) - np.mean(chosen_true),
                       "cấu hình tốt nhất thật sự": true_auc.max()})
print(pd.DataFrame(curse_rows).round(4).to_string(index=False))

- Validation càng **nhỏ** (và số cấu hình thử càng **nhiều**), điểm của người thắng càng lạc quan và người thắng càng hay **không phải** cấu hình tốt nhất thật.
- Vì vậy: chọn cấu hình bằng validation/CV, rồi báo cáo hiệu năng trên **tập test chạm đúng một lần** ở cuối. Nếu nhìn test nhiều lần để chọn, test trở thành validation và cũng bị lạc quan.
- Tracker giúp **trung thực**: nó ghi lại *tất cả* các run (kể cả run thất bại), nên ta biết đã thử bao nhiêu cấu hình — điều kiện để đánh giá con số "tốt nhất" có đáng tin hay không.

## 9. MLflow

MLflow làm đúng những việc trên ở quy mô đội nhóm: lưu params/metrics/tags/artifact, giao diện web so sánh run, model registry (bài 03). Ở đây ta dùng **backend SQLite trong thư mục tạm** — không cần server. (Cần `pip install -e ".[mlops]"`.)

In [ ]:
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")
warnings.filterwarnings("ignore", message="IProgress not found")       # cảnh báo vô hại của tqdm khi thiếu ipywidgets
try:
    import mlflow
except ImportError:
    mlflow = None
    print("Chưa cài MLflow — bỏ qua phần này: pip install -e \".[mlops]\"")

if mlflow is not None:
    import logging
    logging.getLogger("mlflow").setLevel(logging.ERROR)
    mlflow_dir = Path(tempfile.mkdtemp())
    mlflow.set_tracking_uri("sqlite:///" + (mlflow_dir / "mlflow.db").as_posix())
    mlflow.set_experiment("churn-hgb")
    for lr in [0.03, 0.1, 0.3]:
        config = TrainConfig(learning_rate=lr, max_leaf_nodes=8)
        with mlflow.start_run(run_name=f"hgb_lr{lr}"):
            mlflow.log_params(asdict(config))
            model, metrics, data_hash = train(config)
            mlflow.log_metrics(metrics)
            mlflow.set_tags({"data_hash": data_hash, "sklearn": env_snapshot["packages"]["scikit-learn"]})
            mlflow.log_dict(env_snapshot, "environment.json")
    found = mlflow.search_runs(order_by=["metrics.val_auc DESC"])
    print(found[["tags.mlflow.runName", "params.learning_rate", "metrics.val_auc", "tags.data_hash"]].round(4).to_string(index=False))

Xem giao diện: `mlflow ui --backend-store-uri sqlite:///<đường dẫn>/mlflow.db` rồi mở http://127.0.0.1:5000. Một số tính năng khác:
- `mlflow.autolog()` tự log params/metrics/mô hình của sklearn, PyTorch, XGBoost… — tiện, nhưng vẫn nên tự log **hash dữ liệu** và **commit**.
- `mlflow.log_metric(name, value, step=epoch)` cho đường cong học.
- Lựa chọn khác: Weights & Biases, Neptune, Comet, Aim, DVC experiments — cùng khái niệm run/params/metrics/artifact.

## 10. Checklist tái lập

| ✅ | Ghi lại |
|---|---|
| Code | commit hash; không chạy thí nghiệm chính thức khi còn thay đổi chưa commit |
| Dữ liệu | đường dẫn/truy vấn + **hash** nội dung + thời điểm snapshot |
| Cấu hình | mọi siêu tham số, kể cả seed dữ liệu/chia tập/mô hình — dạng dataclass/YAML, không "số ma thuật" trong code |
| Môi trường | lock file hoặc Docker image; phiên bản thư viện trong mỗi run |
| Kết quả | metric kèm **độ biến thiên** (nhiều seed/fold), số cấu hình đã thử, tập test chỉ dùng một lần |

## 🧠 Kiểm tra nhanh

1. Phân biệt lặp lại, tái lập, tái hiện.
2. Vì sao `random_state=0` không làm mô hình "đúng hơn"? Báo cáo kết quả thế nào cho trung thực?
3. Vì sao so sánh theo cặp (cùng lần chia) chính xác hơn so sánh hai trung bình độc lập?
4. Kể hai nguồn bất định ẩn ngoài seed.
5. Một run cần lưu những gì để chạy lại được?
6. Lời nguyền người thắng là gì? Làm sao tránh?

## 📌 Tóm tắt

- Kết quả = code × dữ liệu × cấu hình × môi trường × ngẫu nhiên — ghi lại **cả năm**.
- Seed cho khả năng lặp lại; **độ biến thiên theo seed/lần chia** cho biết kết luận có đáng tin; so sánh mô hình theo cặp.
- Hash dữ liệu, ghi môi trường, cấu hình dạng dataclass; tracker lưu mọi run và phải chạy lại được run từ những gì đã log.
- Chọn trên validation → điểm lạc quan; tập test chạm một lần.

## 📚 Tài liệu tham khảo

- Pineau et al. (2021) — *Improving Reproducibility in Machine Learning Research* (NeurIPS reproducibility checklist)
- Bouthillier et al. (2021) — *Accounting for Variance in Machine Learning Benchmarks*
- Nadeau & Bengio (2003) — *Inference for the Generalization Error* (corrected resampled t-test)
- Cawley & Talbot (2010) — *On Over-fitting in Model Selection and Subsequent Selection Bias in Performance Evaluation*
- Tài liệu MLflow Tracking — mlflow.org/docs/latest/tracking
- PyTorch — *Reproducibility* (pytorch.org/docs/stable/notes/randomness.html)